<a href="https://colab.research.google.com/github/gcallj/test/blob/main/GA_stock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
pip install deap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 8.5 MB/s eta 0:00:00


In [13]:
# -*- coding: utf-8 -*-
"""
GA + Walk-Forward ML (OOS) + Intraday (OHLC) Backtest  — v2 (fixed outputs)
============================================================================

Fixes vs previous version:
1) APPLY "best buy" / "best sell" now uses **next-day OHLC** (i+1) as intended.
   - Signal is computed at end of day i (close), and the suggested order is for day i+1.
   - Columns:
        signal_eod          : signal decided at close(i)
        next_day_filled     : whether the limit would be filled on day i+1
        best_buy_value      : filled price (NaN if not filled)
        best_sell_value     : filled price (NaN if not filled)
        entry_ref_price     : best_* if filled else open(i+1) (optional reference)
        stop/take levels are computed from entry_ref_price.

2) "Score & signal in Excel not working" (all holds / all best_buy == close):
   - We compute suggested entry for next day; last row has no next day -> NaNs as expected.

3) Too many tickers with 0 trades (TEret=0):
   - GA fitness penalizes strategies with very low trades/exposure (prevents "do nothing" winning).
   - GA search ranges for enter_abs are made more permissive (lower thresholds).

4) Cleaner & richer metrics:
   - WF: AUC mean/std, ACC mean, PR-AUC mean, logloss, brier.
   - Trading: return/mdd/sharpe/trades/exposure/win_rate/avg_trade for GA and TEST.
   - Period print per ticker: train/test date ranges.

Expected columns in HISTORY_CSV:
- Date, ticker, open, high, low, close
- plus numeric feature columns.

Outputs:
- CSV: apply_last_{APPLY_DAYS}d__H{FWD_H}.csv
- XLSX: summary_latest + apply_last_{APPLY_DAYS}d

NOTE
- This is research/backtest code. Not financial advice.

Author: ChatGPT (generated)
"""

import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

from deap import base, creator, tools, algorithms

from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss, brier_score_loss


# ==============================================================================
# 0) CONFIG
# ==============================================================================
HISTORY_CSV_PATH = "/content/drive/MyDrive/history_consolidated.csv"
OUTPUT_DIR       = "/content/drive/MyDrive/"

DATE_COL   = "Date"
TICKER_COL = "ticker"

OPEN_COL  = "open"
HIGH_COL  = "high"
LOW_COL   = "low"
CLOSE_COL = "close"

ONLY_SA    = True
LONG_ONLY  = False

APPLY_DAYS = 5
FWD_H = 20

# Regime filter (MA200)
MA_WINDOW = 200
USE_MA_SLOPE_FILTER = True
MA_SLOPE_LOOKBACK = 20
MA_SLOPE_EPS = 0.0
REQUIRE_MA_FOR_ENTRY = True
REQUIRE_MA_FOR_SELL_MA = False

# Z-score (normalizes score_full)
EV_CLIP = 8.0
EV_EMA_SPAN = 5

# ATR
ATR_WINDOW = 14
ATR_MIN_PERIODS = 14
ATR_EPS = 1e-12

# GA ranges
ATR_MULT_RANGE = (1.0, 3.0)
RR_MULT_RANGE  = (1.0, 2.5)

# Friction
COST_BPS     = 2.0
SLIPPAGE_BPS = 2.0

MIN_PRICE     = 0.01
CAP_DAILY_RET = 0.30
CAP_TRADE_RET = 3.00

ONE_YEAR_DAYS = 252
LAMBDA_MDD_1Y = 0.70
MAX_EXPOSURE_1Y = 0.70

# GA hyperparams (reduce for speed)
RANDOM_SEED = 42
GA_POP_SIZE = 220
GA_NGEN     = 55
GA_CX_PB    = 0.70
GA_MUT_PB   = 0.40
GA_TOURN    = 3
EARLY_STOP  = 10

# ML (walk-forward)
WF_SPLITS = 5
ML_RECENCY_HALF_LIFE = 252
ML_RET_CAP = 0.60          # cap fwd return before ATR-normalization
ML_MIN_TRAIN = 260

# Feature selection
MIN_ROWS_TICKER = 450
MIN_FEAT_NONNA_FRAC = 0.60
MIN_FEAT_STD = 1e-12
MAX_FEATURES = 60

# Intraday entry (limit) based on signal strength
ENTRY_DISCOUNT_RANGE = (0.0, 1.5)

# Avoid "do nothing" strategies
GA_MIN_TRADES = 8
GA_MIN_EXPOSURE = 0.05
GA_TRADE_BONUS_PER = 0.02

# Prints
PRINT_EVERY = 1
PRINT_FOLD_DETAILS = False
PRINT_TOP_N = 12
TEST_ONLY_FIRST_10_TICKERS = False
TEST_FIRST_N_TICKERS = 10


# ==============================================================================
# 1) DEAP SETUP
# ==============================================================================
def setup_global_deap():
    if not hasattr(creator, "FitnessMax_PT"):
        creator.create("FitnessMax_PT", base.Fitness, weights=(1.0,))
    if not hasattr(creator, "Individual_PT"):
        creator.create("Individual_PT", list, fitness=creator.FitnessMax_PT)

setup_global_deap()


# ==============================================================================
# 2) DATA CLASSES
# ==============================================================================
@dataclass
class Params:
    enter_abs: float
    exit_abs: float
    atr_mult: float
    rr_mult: float
    entry_discount: float

def sanitize_params(p: Params) -> Params:
    enter_abs = float(max(0.08, p.enter_abs))
    exit_abs  = float(max(0.03, p.exit_abs))
    atr_mult = float(np.clip(p.atr_mult, ATR_MULT_RANGE[0], ATR_MULT_RANGE[1]))
    rr_mult  = float(np.clip(p.rr_mult,  RR_MULT_RANGE[0],  RR_MULT_RANGE[1]))
    entry_discount = float(np.clip(p.entry_discount, ENTRY_DISCOUNT_RANGE[0], ENTRY_DISCOUNT_RANGE[1]))
    if exit_abs >= enter_abs:
        exit_abs = 0.5 * enter_abs
    return Params(enter_abs, exit_abs, atr_mult, rr_mult, entry_discount)


# ==============================================================================
# 3) HELPERS
# ==============================================================================
def _parse_dates_smart(s: pd.Series) -> pd.Series:
    ss = s.astype(str)
    frac_dash = ss.str.contains("-", regex=False).mean()
    if frac_dash > 0.5:
        return pd.to_datetime(ss, errors="coerce", dayfirst=False)
    return pd.to_datetime(ss, errors="coerce", dayfirst=True)

def _sigmoid(x: float) -> float:
    x = float(np.clip(x, -50, 50))
    return float(1.0 / (1.0 + math.exp(-x)))

def add_sma200(df: pd.DataFrame) -> pd.DataFrame:
    df["sma200"] = df.groupby(TICKER_COL, sort=False)[CLOSE_COL].transform(
        lambda s: s.rolling(MA_WINDOW, min_periods=MA_WINDOW).mean()
    )
    if USE_MA_SLOPE_FILTER:
        df["sma200_slope"] = df.groupby(TICKER_COL, sort=False)["sma200"].transform(
            lambda x: (x - x.shift(MA_SLOPE_LOOKBACK)) / float(MA_SLOPE_LOOKBACK)
        )
    else:
        df["sma200_slope"] = np.nan
    return df

def add_atr_ohlc_fast(df: pd.DataFrame) -> pd.DataFrame:
    """
    ATR fast: compute TR with vector ops + groupby rolling mean.
    TR = max(high-low, abs(high-prev_close), abs(low-prev_close))
    """
    h = df[HIGH_COL].astype(float)
    l = df[LOW_COL].astype(float)
    c = df[CLOSE_COL].astype(float)
    pc = df.groupby(TICKER_COL, sort=False)[CLOSE_COL].shift(1).astype(float)

    tr1 = (h - l).abs()
    tr2 = (h - pc).abs()
    tr3 = (l - pc).abs()

    tr = np.nanmax(np.vstack([tr1.to_numpy(), tr2.to_numpy(), tr3.to_numpy()]), axis=0)
    tr = pd.Series(tr, index=df.index)

    df["atr"] = tr.groupby(df[TICKER_COL], sort=False).transform(
        lambda s: s.rolling(ATR_WINDOW, min_periods=ATR_MIN_PERIODS).mean()
    )
    return df

def regime_ok(sig: int, price: float, sma200: float, sma_slope: float) -> bool:
    # MA rule
    if sig > 0:
        if REQUIRE_MA_FOR_ENTRY:
            if (sma200 is None) or (not np.isfinite(sma200)):
                return False
            ok_ma = (price > sma200)
        else:
            ok_ma = True
    else:
        if REQUIRE_MA_FOR_SELL_MA and REQUIRE_MA_FOR_ENTRY:
            if (sma200 is None) or (not np.isfinite(sma200)):
                return False
            ok_ma = (price < sma200)
        else:
            ok_ma = True

    # slope rule
    if USE_MA_SLOPE_FILTER:
        if (sma_slope is None) or (not np.isfinite(sma_slope)):
            return False
        ok_sl = (sma_slope > MA_SLOPE_EPS) if sig > 0 else (sma_slope < -MA_SLOPE_EPS)
    else:
        ok_sl = True

    return bool(ok_ma and ok_sl)

def buyhold_capped(close: np.ndarray) -> float:
    n = len(close)
    if n < 2:
        return 0.0
    log_eq = 0.0
    for i in range(1, n):
        pr0, pr1 = float(close[i-1]), float(close[i])
        if pr0 <= MIN_PRICE or pr1 <= MIN_PRICE:
            continue
        daily = (pr1/pr0) - 1.0
        daily = float(np.clip(daily, -CAP_DAILY_RET, CAP_DAILY_RET))
        log_eq += math.log1p(daily)
    return float(math.exp(log_eq) - 1.0)

def fitness_return_1y(stats_1y: Dict[str, float]) -> float:
    ret = float(stats_1y["total_return"])
    mdd_abs = abs(float(stats_1y["mdd"]))
    expo = float(stats_1y["exposure"])
    n_tr = float(stats_1y["n_trades"])
    sh = float(stats_1y["sharpe"])
    wr = float(stats_1y.get("win_rate", 0.0))

    f = ret - LAMBDA_MDD_1Y * mdd_abs + 0.10 * sh

    if expo > MAX_EXPOSURE_1Y:
        f -= (expo - MAX_EXPOSURE_1Y) * 1.5
    if expo < GA_MIN_EXPOSURE:
        f -= (GA_MIN_EXPOSURE - expo) * 2.0

    if n_tr < GA_MIN_TRADES:
        deficit = GA_MIN_TRADES - n_tr
        f -= 0.10 * deficit + 0.05 * (deficit ** 2)

    f += min(n_tr, 30.0) * GA_TRADE_BONUS_PER
    if n_tr >= GA_MIN_TRADES:
        f += max(0.0, wr - 0.5) * 0.15

    return float(f)

def make_signal_eod(score_ev_eod: float, enter_abs: float, close_eod: float, sma200_eod: float, slope_eod: float) -> str:
    """
    Signal decided at end of day i (to be acted on day i+1).
    """
    if (not np.isfinite(score_ev_eod)) or (not np.isfinite(enter_abs)) or (enter_abs <= 0) or (not np.isfinite(close_eod)):
        return "hold"
    if score_ev_eod >= enter_abs and regime_ok(+1, close_eod, sma200_eod, slope_eod):
        return "buy"
    if (not LONG_ONLY) and (score_ev_eod <= -enter_abs) and regime_ok(-1, close_eod, sma200_eod, slope_eod):
        return "sell"
    return "hold"

def score_0_100_from_ev(
    score_ev: float,
    recent_scores_ev: np.ndarray,
    quality: float,
) -> float:
    if (not np.isfinite(score_ev)):
        return 50.0
    recent = recent_scores_ev[np.isfinite(recent_scores_ev)] if recent_scores_ev is not None else np.array([], dtype=np.float64)
    if len(recent) == 0:
        pct_rank = 0.5
    else:
        pct_rank = float(np.mean(recent <= score_ev))
    base_score = 2.0 + 96.0 * pct_rank
    tilt_factor = 0.3 + 0.7 * float(np.clip(quality, 0.0, 1.0))
    return float(np.clip(50.0 + tilt_factor * (base_score - 50.0), 0.0, 100.0))

def compute_quality_factor(test_sharpe: float, test_return: float, trades_1y: float) -> float:
    q_sh = _sigmoid((float(test_sharpe) - 0.10) / 0.30) if np.isfinite(test_sharpe) else 0.5
    q_ret = _sigmoid(float(test_return) / 0.15) if np.isfinite(test_return) else 0.5
    q_tr = _sigmoid((float(trades_1y) - 8.0) / 4.0) if np.isfinite(trades_1y) else 0.3
    return float(np.clip(0.45 * q_sh + 0.35 * q_ret + 0.20 * q_tr, 0.0, 1.0))

def compute_wf_quality(wf_auc_mean: float, wf_ap_mean: float, wf_auc_std: float) -> float:
    q_auc = np.clip((float(wf_auc_mean) - 0.50) / 0.18, 0.0, 1.0) if np.isfinite(wf_auc_mean) else 0.0
    q_ap = np.clip((float(wf_ap_mean) - 0.50) / 0.20, 0.0, 1.0) if np.isfinite(wf_ap_mean) else 0.0
    q_stab = 1.0 - np.clip(float(wf_auc_std) / 0.12, 0.0, 1.0) if np.isfinite(wf_auc_std) else 0.0
    return float(np.clip(0.45 * q_auc + 0.35 * q_ap + 0.20 * q_stab, 0.0, 1.0))

def adjust_params_by_wf_quality(p: Params, wf_quality: float) -> Params:
    p = sanitize_params(p)
    q = float(np.clip(wf_quality, 0.0, 1.0))
    # weaker models become more selective and ask better entry prices
    sel_mult = 1.0 + (1.0 - q) * 0.55
    enter_abs = p.enter_abs * sel_mult
    exit_abs = min(enter_abs * 0.85, p.exit_abs * (1.0 + (1.0 - q) * 0.25))
    entry_discount = p.entry_discount * (1.0 + (1.0 - q) * 0.60)
    return sanitize_params(Params(enter_abs, exit_abs, p.atr_mult, p.rr_mult, entry_discount))

def compute_model_entry_price(o1: float, atr1: float, score_ev_eod: float, enter_abs: float, side: int, entry_discount: float) -> float:
    if (not np.isfinite(o1)) or (not np.isfinite(atr1)) or atr1 <= ATR_EPS:
        return float("nan")
    if (not np.isfinite(score_ev_eod)) or (not np.isfinite(enter_abs)) or enter_abs <= 0:
        return float("nan")

    strength_ratio = abs(float(score_ev_eod)) / float(enter_abs)
    strength_ratio = max(strength_ratio, 0.5)
    inv_strength = float(np.clip(1.0 / strength_ratio, 0.3, 2.0))
    discount_atr = float(entry_discount * inv_strength * atr1)
    return float(o1 - discount_atr) if side > 0 else float(o1 + discount_atr)

def compute_levels_from_atr(entry_price: float, atr_val: float, p: Params):
    """
    Returns: stop_abs, take_abs, stop_pct, take_pct, buy_entry, buy_stop, buy_take, sell_entry, sell_stop, sell_take
    """
    if (not np.isfinite(entry_price)) or (entry_price <= 0) or (not np.isfinite(atr_val)) or (atr_val <= ATR_EPS):
        return (np.nan, np.nan, np.nan, np.nan,
                np.nan, np.nan, np.nan,
                np.nan, np.nan, np.nan)

    p = sanitize_params(p)
    stop_abs = float(p.atr_mult * atr_val)
    take_abs = float(p.rr_mult  * stop_abs)

    stop_pct = float(stop_abs / max(entry_price, 1e-12))
    take_pct = float(take_abs / max(entry_price, 1e-12))

    buy_entry = float(entry_price)
    buy_stop  = float(entry_price - stop_abs)
    buy_take  = float(entry_price + take_abs)

    sell_entry = float(entry_price)
    sell_stop  = float(entry_price + stop_abs)
    sell_take  = float(entry_price - take_abs)

    return (stop_abs, take_abs, stop_pct, take_pct,
            buy_entry, buy_stop, buy_take,
            sell_entry, sell_stop, sell_take)

def nextday_limit_fill(o1: float, h1: float, l1: float, atr1: float, score_ev_eod: float, enter_abs: float, side: int, entry_discount: float) -> Tuple[bool, float, float]:
    if (not np.isfinite(o1)) or (not np.isfinite(h1)) or (not np.isfinite(l1)) or (not np.isfinite(atr1)) or atr1 <= ATR_EPS:
        return (False, np.nan, np.nan)

    limit = compute_model_entry_price(o1, atr1, score_ev_eod, enter_abs, side, entry_discount)
    if not np.isfinite(limit):
        return (False, np.nan, np.nan)

    if side > 0:
        if float(o1) <= limit:
            return (True, float(o1), float(limit))
        if float(l1) <= limit <= float(h1):
            return (True, float(limit), float(limit))
        return (False, np.nan, float(limit))
    else:
        if float(o1) >= limit:
            return (True, float(o1), float(limit))
        if float(l1) <= limit <= float(h1):
            return (True, float(limit), float(limit))
        return (False, np.nan, float(limit))

def make_recency_weights(n: int, half_life: int) -> np.ndarray:
    if n <= 1:
        return np.ones(n, dtype=np.float64)
    lam = math.log(2.0) / max(1.0, float(half_life))
    idx = np.arange(n, dtype=np.float64)
    w = np.exp(-lam * ((n - 1) - idx))
    w = w / max(1e-12, float(np.mean(w)))
    return w.astype(np.float64)

def _safe_pr_auc(y_true_bin: np.ndarray, y_prob: np.ndarray) -> float:
    try:
        from sklearn.metrics import average_precision_score
        return float(average_precision_score(y_true_bin, y_prob))
    except Exception:
        return float("nan")


# ==============================================================================
# 4) ML (Walk-Forward OOS probabilities)
# ==============================================================================
def get_clean_walk_forward_predictions(
    X: np.ndarray,
    y_bin: np.ndarray,
    y_mag: np.ndarray,
    w: np.ndarray,
    dates: np.ndarray,
    ticker: str,
    n_splits: int = WF_SPLITS
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Dict[str, float]]:
    tscv = TimeSeriesSplit(n_splits=n_splits)

    oos_prob = np.full(len(y_bin), np.nan, dtype=np.float64)
    oos_mag = np.full(len(y_bin), np.nan, dtype=np.float64)

    aucs, accs, briers, loglosses, ap_scores = [], [], [], [], []
    fold_ranges = []

    for fold_i, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
        X_tr, y_tr = X[train_idx], y_bin[train_idx]
        X_te, y_te = X[test_idx], y_bin[test_idx]
        mag_tr = y_mag[train_idx]
        w_tr = w[train_idx] if w is not None else None

        clf = HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=240,
            max_depth=5,
            l2_regularization=1.2,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=10,
            random_state=RANDOM_SEED,
        )
        reg = HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_iter=240,
            max_depth=5,
            l2_regularization=1.2,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=10,
            random_state=RANDOM_SEED,
        )
        clf.fit(X_tr, y_tr, sample_weight=w_tr)
        reg.fit(X_tr, mag_tr, sample_weight=w_tr)

        p_te = clf.predict_proba(X_te)[:, 1].astype(np.float64)
        m_te = np.clip(reg.predict(X_te).astype(np.float64), 0.0, None)
        oos_prob[test_idx] = p_te
        oos_mag[test_idx] = m_te

        train_start = pd.to_datetime(dates[train_idx[0]]).date()
        train_end = pd.to_datetime(dates[train_idx[-1]]).date()
        test_start = pd.to_datetime(dates[test_idx[0]]).date()
        test_end = pd.to_datetime(dates[test_idx[-1]]).date()
        fold_msg = f"[{ticker}] Fold {fold_i}/{n_splits}: TRAIN {train_start} -> {train_end} ({len(train_idx)} rows) | TEST {test_start} -> {test_end} ({len(test_idx)} rows)"
        if PRINT_FOLD_DETAILS:
            print("  " + fold_msg)
        fold_ranges.append(f"{train_start}|{train_end}|{test_start}|{test_end}")

        try:
            auc = roc_auc_score(y_te, p_te)
        except Exception:
            auc = 0.5
        yhat = (p_te >= 0.5).astype(int)
        acc = accuracy_score(y_te, yhat)
        try:
            ll = log_loss(y_te, np.clip(p_te, 1e-6, 1-1e-6))
        except Exception:
            ll = float("nan")
        try:
            br = brier_score_loss(y_te, p_te)
        except Exception:
            br = float("nan")
        ap = _safe_pr_auc(y_te, p_te)

        aucs.append(float(auc)); accs.append(float(acc)); loglosses.append(float(ll)); briers.append(float(br)); ap_scores.append(float(ap) if np.isfinite(ap) else float("nan"))

    direction_sign = 2.0 * oos_prob - 1.0
    oos_ev = direction_sign * oos_mag

    metrics = {
        "wf_auc_mean": float(np.nanmean(aucs)) if len(aucs) else float("nan"),
        "wf_auc_std":  float(np.nanstd(aucs)) if len(aucs) else float("nan"),
        "wf_acc_mean": float(np.nanmean(accs)) if len(accs) else float("nan"),
        "wf_logloss":  float(np.nanmean(loglosses)) if len(loglosses) else float("nan"),
        "wf_brier":    float(np.nanmean(briers)) if len(briers) else float("nan"),
        "wf_ap_mean":  float(np.nanmean(ap_scores)) if len(ap_scores) else float("nan"),
        "wf_n_folds":  int(len(aucs)),
        "wf_fold_ranges": ';'.join(fold_ranges),
    }
    return oos_ev, oos_prob, oos_mag, metrics


# ==============================================================================
# 5) BACKTEST (Intraday OHLC-aware)
# ==============================================================================
def backtest_stats_only_intraday(o, h, l, c, score_ev, sma200, sma_slope, atr, p: Params) -> Dict[str, float]:
    """
    Trade enters on day i using score_z[i-1] (signal from previous close).
    """
    p = sanitize_params(p)
    n = len(c)
    if n < 3:
        return {"total_return":0.0,"mdd":0.0,"sharpe":0.0,"n_trades":0.0,"win_rate":0.0,"avg_trade":0.0,"exposure":0.0}

    cost_leg = (1.0 - (COST_BPS + SLIPPAGE_BPS)/10000.0)
    log_cost = math.log(max(cost_leg, 1e-12))

    log_eq   = np.zeros(n, dtype=np.float64)
    log_rets = np.zeros(n, dtype=np.float64)

    pos = 0
    entry_price = 0.0
    stop_abs = 0.0
    take_abs = 0.0

    n_trades = 0
    n_wins = 0
    trade_sum = 0.0
    pos_days = 0

    for i in range(1, n):
        o1 = float(o[i]); h1 = float(h[i]); l1 = float(l[i]); c1 = float(c[i])
        c0 = float(c[i-1])

        if c0 <= MIN_PRICE or c1 <= MIN_PRICE:
            log_eq[i] = log_eq[i-1]
            continue

        daily = (c1/c0) - 1.0
        daily = float(np.clip(daily, -CAP_DAILY_RET, CAP_DAILY_RET))
        step_ret = (pos * daily) if pos != 0 else 0.0
        log_eq[i] = log_eq[i-1] + math.log1p(step_ret)
        log_rets[i] = log_eq[i] - log_eq[i-1]
        if pos != 0:
            pos_days += 1

        # EXIT (intraday)
        if pos != 0:
            exited = False
            exit_px = np.nan

            if pos > 0:
                stop_px = entry_price - stop_abs
                take_px = entry_price + take_abs

                if o1 <= stop_px:
                    exited, exit_px = True, o1
                elif o1 >= take_px:
                    exited, exit_px = True, o1
                else:
                    hit_stop = (l1 <= stop_px)
                    hit_take = (h1 >= take_px)
                    if hit_stop and hit_take:
                        exited, exit_px = True, stop_px  # conservative
                    elif hit_stop:
                        exited, exit_px = True, stop_px
                    elif hit_take:
                        exited, exit_px = True, take_px

                if not exited:
                    if abs(float(score_ev[i])) < p.exit_abs:
                        exited, exit_px = True, c1
                    if REQUIRE_MA_FOR_ENTRY:
                        ma_ = float(sma200[i])
                        if np.isfinite(ma_) and (c1 < ma_):
                            exited, exit_px = True, c1

                if exited:
                    trade_ret = (exit_px / max(entry_price, 1e-12)) - 1.0
                    trade_ret = float(np.clip(trade_ret, -CAP_TRADE_RET, CAP_TRADE_RET))

                    log_eq[i] += log_cost
                    log_rets[i] = log_eq[i] - log_eq[i-1]

                    net_trade = (1.0 + trade_ret) * (cost_leg**2) - 1.0
                    n_trades += 1
                    if net_trade > 0:
                        n_wins += 1
                    trade_sum += net_trade

                    pos = 0
                    entry_price = 0.0
                    stop_abs = 0.0
                    take_abs = 0.0
                    continue

            else:
                stop_px = entry_price + stop_abs
                take_px = entry_price - take_abs

                if o1 >= stop_px:
                    exited, exit_px = True, o1
                elif o1 <= take_px:
                    exited, exit_px = True, o1
                else:
                    hit_stop = (h1 >= stop_px)
                    hit_take = (l1 <= take_px)
                    if hit_stop and hit_take:
                        exited, exit_px = True, stop_px
                    elif hit_stop:
                        exited, exit_px = True, stop_px
                    elif hit_take:
                        exited, exit_px = True, take_px

                if not exited:
                    if abs(float(score_ev[i])) < p.exit_abs:
                        exited, exit_px = True, c1
                    if REQUIRE_MA_FOR_ENTRY:
                        ma_ = float(sma200[i])
                        if np.isfinite(ma_) and (c1 > ma_):
                            exited, exit_px = True, c1

                if exited:
                    trade_ret = (entry_price / max(exit_px, 1e-12)) - 1.0  # short
                    trade_ret = float(np.clip(trade_ret, -CAP_TRADE_RET, CAP_TRADE_RET))

                    log_eq[i] += log_cost
                    log_rets[i] = log_eq[i] - log_eq[i-1]

                    net_trade = (1.0 + trade_ret) * (cost_leg**2) - 1.0
                    n_trades += 1
                    if net_trade > 0:
                        n_wins += 1
                    trade_sum += net_trade

                    pos = 0
                    entry_price = 0.0
                    stop_abs = 0.0
                    take_abs = 0.0
                    continue

        # ENTRY (uses score from day i-1, enters day i)
        if pos == 0 and i >= 1:
            s_prev = float(score_ev[i-1]) if np.isfinite(score_ev[i-1]) else np.nan
            sig = 0
            if np.isfinite(s_prev):
                if s_prev >= p.enter_abs:
                    sig = 1
                elif (not LONG_ONLY) and (s_prev <= -p.enter_abs):
                    sig = -1

            if sig != 0:
                c_prev = float(c[i-1])
                ma_prev = float(sma200[i-1]) if np.isfinite(sma200[i-1]) else np.nan
                sl_prev = float(sma_slope[i-1]) if np.isfinite(sma_slope[i-1]) else np.nan
                if not regime_ok(sig, c_prev, ma_prev, sl_prev):
                    continue

                atr_i = float(atr[i]) if np.isfinite(atr[i]) else np.nan
                if (not np.isfinite(atr_i)) or atr_i <= ATR_EPS:
                    continue

                filled, fill_px, _ = nextday_limit_fill(o1, h1, l1, atr_i, s_prev, p.enter_abs, sig, p.entry_discount)
                if not filled:
                    continue

                pos = sig
                entry_price = float(fill_px)
                stop_abs = float(p.atr_mult * atr_i)
                take_abs = float(p.rr_mult * stop_abs)

                log_eq[i] += log_cost
                log_rets[i] = log_eq[i] - log_eq[i-1]

    exposure = float(pos_days / max(n-1, 1))
    total_return = float(math.exp(log_eq[-1] - log_eq[0]) - 1.0)

    peak = np.maximum.accumulate(log_eq)
    dd = np.exp(log_eq - peak) - 1.0
    mdd = float(np.min(dd))

    mu = float(np.nanmean(log_rets))
    sd = float(np.nanstd(log_rets, ddof=1))
    sharpe = (mu / sd) * math.sqrt(252.0) if (sd > 1e-9) else 0.0

    win_rate = (n_wins / n_trades) if n_trades > 0 else 0.0
    avg_trade = (trade_sum / n_trades) if n_trades > 0 else 0.0

    return {
        "total_return": total_return,
        "mdd": mdd,
        "sharpe": sharpe,
        "n_trades": float(n_trades),
        "win_rate": float(win_rate),
        "avg_trade": float(avg_trade),
        "exposure": float(exposure),
    }


# ==============================================================================
# 6) GA optimize strategy only (score_z fixed)
# ==============================================================================
def infer_ga_ranges(score_z: np.ndarray) -> Tuple[Tuple[float, float], Tuple[float, float]]:
    x = np.abs(score_z[np.isfinite(score_z)])
    if len(x) < 200:
        return (0.15, 1.80), (0.04, 0.80)

    q30 = float(np.quantile(x, 0.30))
    q80 = float(np.quantile(x, 0.80))
    q92 = float(np.quantile(x, 0.92))

    enter_lo = max(0.08, q30 * 0.85)
    enter_hi = max(enter_lo * 2.5, q92 * 1.2, 0.60)

    exit_lo = 0.04
    exit_hi = max(exit_lo, min(enter_lo * 0.9, 1.00))
    return (enter_lo, enter_hi), (exit_lo, exit_hi)

def ga_optimize_strategy_only(o, h, l, c, score_ev, ma, sl, atr, train_idx: np.ndarray):
    o_tr = o[train_idx]; h_tr = h[train_idx]; l_tr = l[train_idx]; c_tr = c[train_idx]
    z_tr = score_ev[train_idx]
    ma_tr = ma[train_idx]; sl_tr = sl[train_idx]; atr_tr = atr[train_idx]

    if np.isfinite(z_tr).sum() < 120:
        return None

    n_tr = len(c_tr)
    split_int = int(n_tr * 0.70)
    idx_A = slice(0, split_int)
    idx_B = slice(split_int, n_tr)

    def fitness_internal(p_: Params) -> float:
        p_ = sanitize_params(p_)

        def eval_block(o_, h_, l_, c_, z_, ma_, sl_, atr_):
            if len(c_) < 80:
                return -1e9
            st = backtest_stats_only_intraday(o_, h_, l_, c_, z_, ma_, sl_, atr_, p_)
            return fitness_return_1y(st)

        fA = eval_block(o_tr[idx_A], h_tr[idx_A], l_tr[idx_A], c_tr[idx_A], z_tr[idx_A], ma_tr[idx_A], sl_tr[idx_A], atr_tr[idx_A])
        fB = eval_block(o_tr[idx_B], h_tr[idx_B], l_tr[idx_B], c_tr[idx_B], z_tr[idx_B], ma_tr[idx_B], sl_tr[idx_B], atr_tr[idx_B])
        return min(fA, fB)

    (enter_lo, enter_hi), (exit_lo, exit_hi) = infer_ga_ranges(z_tr)

    toolbox = base.Toolbox()
    toolbox.register("attr_enter", random.uniform, enter_lo, enter_hi)
    toolbox.register("attr_exit",  random.uniform, exit_lo,  exit_hi)
    toolbox.register("attr_atr",   random.uniform, ATR_MULT_RANGE[0], ATR_MULT_RANGE[1])
    toolbox.register("attr_rr",    random.uniform, RR_MULT_RANGE[0],  RR_MULT_RANGE[1])
    toolbox.register("attr_entry_discount", random.uniform, ENTRY_DISCOUNT_RANGE[0], ENTRY_DISCOUNT_RANGE[1])

    toolbox.register("individual", tools.initCycle, creator.Individual_PT,
                     (toolbox.attr_enter, toolbox.attr_exit, toolbox.attr_atr, toolbox.attr_rr, toolbox.attr_entry_discount), n=1)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("select", tools.selTournament, tournsize=GA_TOURN)
    toolbox.register("mate", tools.cxBlend, alpha=0.30)

    def mutate_gaussian(ind):
        for j in range(len(ind)):
            if random.random() < 0.25:
                ind[j] = float(ind[j]) + random.gauss(0.0, 0.18)
        return (ind,)

    toolbox.register("mutate", mutate_gaussian)
    toolbox.register("evaluate", lambda ind: (fitness_internal(Params(*ind)),))

    pop = toolbox.population(n=GA_POP_SIZE)
    hof = tools.HallOfFame(1)

    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)
    hof.update(pop)

    best_fit = hof[0].fitness.values[0]
    no_improve = 0

    for _gen in range(1, GA_NGEN + 1):
        offspring = algorithms.varAnd(pop, toolbox, cxpb=GA_CX_PB, mutpb=GA_MUT_PB)
        for ind in offspring:
            ind.fitness.values = toolbox.evaluate(ind)
        pop[:] = offspring
        hof.update(pop)

        cur = hof[0].fitness.values[0]
        if cur > best_fit + 1e-9:
            best_fit = cur
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= EARLY_STOP:
            break

    best_p = sanitize_params(Params(*hof[0]))

    start = max(0, len(c_tr) - ONE_YEAR_DAYS)
    stats_train = backtest_stats_only_intraday(
        o_tr[start:], h_tr[start:], l_tr[start:], c_tr[start:], z_tr[start:],
        ma_tr[start:], sl_tr[start:], atr_tr[start:], best_p
    )
    fit1y = fitness_return_1y(stats_train)
    bh1y  = buyhold_capped(c_tr[start:])

    return best_p, stats_train, fit1y, bh1y


# ==============================================================================
# 7) LOAD
# ==============================================================================
def load_full_history_all_cols(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={TICKER_COL:"string"}, low_memory=False)
    df[TICKER_COL] = df[TICKER_COL].astype("string").str.strip()
    df[DATE_COL] = _parse_dates_smart(df[DATE_COL])

    for col in [OPEN_COL, HIGH_COL, LOW_COL, CLOSE_COL]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=[DATE_COL, TICKER_COL, OPEN_COL, HIGH_COL, LOW_COL, CLOSE_COL])
    df = df[(df[CLOSE_COL] > MIN_PRICE) & (df[OPEN_COL] > MIN_PRICE) & (df[HIGH_COL] > MIN_PRICE) & (df[LOW_COL] > MIN_PRICE)]
    if ONLY_SA:
        df = df[df[TICKER_COL].str.endswith(".SA", na=False)]

    df = df.sort_values([TICKER_COL, DATE_COL]).reset_index(drop=True)
    df = add_sma200(df)
    df = add_atr_ohlc_fast(df)
    return df


# ==============================================================================
# 8) RUN
# ==============================================================================
def _pct(x):
    return f"{x*100:7.1f}%" if np.isfinite(x) else "    nan "
def _flt(x, w=6, p=3):
    return f"{x:{w}.{p}f}" if np.isfinite(x) else f"{'nan':>{w}}"

def run():
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    df = load_full_history_all_cols(HISTORY_CSV_PATH)

    exclude = {DATE_COL, TICKER_COL, OPEN_COL, HIGH_COL, LOW_COL, CLOSE_COL, "sma200", "sma200_slope", "atr"}
    num_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
    base_feat_cols = num_cols[:MAX_FEATURES]

    tickers = df[TICKER_COL].dropna().unique().tolist()
    total_tickers = len(tickers)
    if TEST_ONLY_FIRST_10_TICKERS:
        tickers = tickers[:TEST_FIRST_N_TICKERS]

    print(f"[FEATS] numeric candidates: {len(base_feat_cols)} (using up to {MAX_FEATURES})")
    if TEST_ONLY_FIRST_10_TICKERS:
        print(f"[TICKERS] processing first {len(tickers)} of {total_tickers} (TEST_ONLY_FIRST_10_TICKERS=True)")
    else:
        print(f"[TICKERS] total: {len(tickers)}")

    header = (
        f"{'#':>4} | {'TICKER':<10} | "
        f"{'AUC':>6} {'AUCsd':>6} {'ACC':>5} {'AP':>5} | "
        f"{'GAret':>7} {'GAmdd':>7} {'GAsh':>6} {'TR':>4} {'EX':>4} {'BH':>7} || "
        f"{'TEret':>7} {'TEmdd':>7} {'TEsh':>6} {'TEtr':>4} | {'TRAIN_RANGE':<23} {'TEST_RANGE':<23}"
    )
    print("\n" + header + "\n" + "-"*len(header))

    reasons: Dict[str, int] = {}
    results_summary: List[Dict] = []
    results_apply: List[Dict] = []
    prog = 0

    for tkr in tickers:
        g = df[df[TICKER_COL] == tkr].copy().sort_values(DATE_COL)

        if len(g) < MIN_ROWS_TICKER:
            reasons["short_len"] = reasons.get("short_len", 0) + 1
            continue

        # stable per-ticker feature selection
        feat_cols: List[str] = []
        for ccol in base_feat_cols:
            s = pd.to_numeric(g[ccol], errors="coerce")
            if float(s.notna().mean()) < MIN_FEAT_NONNA_FRAC:
                continue
            if float(s.std(skipna=True)) <= MIN_FEAT_STD:
                continue
            feat_cols.append(ccol)

        if len(feat_cols) < 5:
            reasons["few_feats"] = reasons.get("few_feats", 0) + 1
            continue

        # arrays
        dates = pd.to_datetime(g[DATE_COL], errors="coerce").to_numpy()
        o = g[OPEN_COL].to_numpy(np.float64)
        h = g[HIGH_COL].to_numpy(np.float64)
        l = g[LOW_COL].to_numpy(np.float64)
        c = g[CLOSE_COL].to_numpy(np.float64)
        ma  = g["sma200"].to_numpy(np.float64)
        sl  = g["sma200_slope"].to_numpy(np.float64)
        atr = g["atr"].to_numpy(np.float64)

        # forward return target for labeling
        ret_fwd = (np.roll(c, -FWD_H) - c) / np.maximum(c, 1e-12)
        ret_fwd[-FWD_H:] = np.nan

        atr_pct = atr / np.maximum(c, 1e-12)
        y_atr_norm = np.clip(ret_fwd, -ML_RET_CAP, ML_RET_CAP) / np.maximum(atr_pct, 1e-6)
        y_atr_norm[~np.isfinite(atr_pct)] = np.nan
        y_atr_norm[atr_pct <= 1e-6] = np.nan

        y_bin = (y_atr_norm > 0.0).astype(int)
        y_mag = np.abs(y_atr_norm)

        X = g[feat_cols].apply(pd.to_numeric, errors="coerce").to_numpy(np.float64)

        valid_mask = np.isfinite(o) & np.isfinite(h) & np.isfinite(l) & np.isfinite(c) & np.isfinite(y_atr_norm)
        if valid_mask.sum() < ML_MIN_TRAIN:
            reasons["few_valid"] = reasons.get("few_valid", 0) + 1
            continue

        valid_idx = np.where(valid_mask)[0]
        Xv = X[valid_idx]
        yv = y_bin[valid_idx].astype(int)
        wv = make_recency_weights(len(yv), ML_RECENCY_HALF_LIFE)

        # Walk-forward OOS score
        if PRINT_FOLD_DETAILS:
            print(f"\n[{tkr}] walk-forward folds")
        try:
            oos_ev, oos_prob, oos_mag, wf_m = get_clean_walk_forward_predictions(Xv, yv, y_mag[valid_idx], wv, dates[valid_idx], tkr, n_splits=WF_SPLITS)
        except Exception:
            reasons["wf_error"] = reasons.get("wf_error", 0) + 1
            continue

        score_full = np.full(len(c), np.nan, dtype=np.float64)
        score_full[valid_idx] = oos_ev

        score_ev_raw = np.clip(score_full, -EV_CLIP, EV_CLIP)
        score_ev = pd.Series(score_ev_raw).ewm(span=EV_EMA_SPAN, adjust=False, min_periods=1).mean().to_numpy(np.float64)
        score_ev = np.clip(score_ev, -EV_CLIP, EV_CLIP)

        has_z_idx = np.where(np.isfinite(score_ev))[0]
        if len(has_z_idx) < 252:
            reasons["short_oos"] = reasons.get("short_oos", 0) + 1
            continue

        split_point = int(len(has_z_idx) * 0.80)
        ga_train_idx = has_z_idx[:split_point]
        test_final_idx = has_z_idx[split_point:]

        out_ga = ga_optimize_strategy_only(o, h, l, c, score_ev, ma, sl, atr, ga_train_idx)
        if out_ga is None:
            reasons["ga_fail"] = reasons.get("ga_fail", 0) + 1
            continue

        best_p, stats_ga_train, fit_ga, bh_ga = out_ga

        wf_quality = compute_wf_quality(wf_m.get("wf_auc_mean", np.nan), wf_m.get("wf_ap_mean", np.nan), wf_m.get("wf_auc_std", np.nan))
        best_p_adj = adjust_params_by_wf_quality(best_p, wf_quality)

        te_stats = backtest_stats_only_intraday(
            o[test_final_idx], h[test_final_idx], l[test_final_idx], c[test_final_idx],
            score_ev[test_final_idx], ma[test_final_idx], sl[test_final_idx], atr[test_final_idx],
            best_p_adj
        )

        # Print periods (train vs test) based on indices used
        train_rng = (pd.to_datetime(dates[ga_train_idx[0]]).date(), pd.to_datetime(dates[ga_train_idx[-1]]).date())
        test_rng  = (pd.to_datetime(dates[test_final_idx[0]]).date(), pd.to_datetime(dates[test_final_idx[-1]]).date())

        prog += 1
        if (prog % PRINT_EVERY) == 0:
            print(
                f"{prog:4d} | {tkr:<10} | "
                f"{_flt(wf_m['wf_auc_mean'],6,3)} {_flt(wf_m['wf_auc_std'],6,3)} {_flt(wf_m['wf_acc_mean'],5,3)} {_flt(wf_m['wf_ap_mean'],5,3)} | "
                f"{_pct(stats_ga_train['total_return'])} {_pct(stats_ga_train['mdd'])} {_flt(stats_ga_train['sharpe'],6,2)} "
                f"{int(stats_ga_train['n_trades']):4d} {int(stats_ga_train['exposure']*100):3d}% {_pct(bh_ga)} || "
                f"{_pct(te_stats.get('total_return', np.nan))} {_pct(te_stats.get('mdd', np.nan))} {_flt(te_stats.get('sharpe', np.nan),6,2)} "
                f"{int(te_stats.get('n_trades', 0) if np.isfinite(te_stats.get('n_trades', np.nan)) else 0):4d} | "
                f"{str(train_rng[0])+' -> '+str(train_rng[1]):<23} {str(test_rng[0])+' -> '+str(test_rng[1]):<23}"
            )

        # APPLY: last APPLY_DAYS *decision days*; suggested trade is for next day (i+1)
        start_apply = max(0, len(g) - APPLY_DAYS)
        idx_apply = np.arange(start_apply, len(g))

        for i in idx_apply:
            # EOD decision (day i)
            date_i = pd.to_datetime(dates[i])
            close_i = float(c[i])
            ev_i = float(score_ev[i]) if np.isfinite(score_ev[i]) else np.nan
            atr_i = float(atr[i]) if np.isfinite(atr[i]) else np.nan
            ma_i = float(ma[i]) if np.isfinite(ma[i]) else np.nan
            sl_i = float(sl[i]) if np.isfinite(sl[i]) else np.nan

            signal_eod = make_signal_eod(ev_i, best_p_adj.enter_abs, close_i, ma_i, sl_i)
            final_signal = signal_eod
            quality = 0.60 * wf_quality + 0.40 * compute_quality_factor(float(te_stats.get("sharpe", np.nan)), float(te_stats.get("total_return", np.nan)), float(stats_ga_train.get("n_trades", np.nan)))
            recent_scores = score_ev[max(0, i-503): i+1]
            score100 = score_0_100_from_ev(ev_i, recent_scores, quality)

            # next day projections + fills (always compute projected limits when i+1 exists)
            best_buy_value = np.nan
            best_sell_value = np.nan
            limit_price = np.nan
            next_day_filled = False
            next_day_filled_buy = False
            next_day_filled_sell = False
            projected_buy_limit = np.nan
            projected_sell_limit = np.nan
            entry_ref_price = np.nan

            if (i + 1) < len(g):
                o1, h1, l1 = float(o[i+1]), float(h[i+1]), float(l[i+1])
                atr1 = float(atr[i+1]) if np.isfinite(atr[i+1]) else np.nan

                projected_buy_limit = compute_model_entry_price(o1, atr1, ev_i, best_p_adj.enter_abs, +1, best_p_adj.entry_discount)
                projected_sell_limit = compute_model_entry_price(o1, atr1, ev_i, best_p_adj.enter_abs, -1, best_p_adj.entry_discount)

                fb, fill_b, lim_b = nextday_limit_fill(o1, h1, l1, atr1, ev_i, best_p_adj.enter_abs, +1, best_p_adj.entry_discount)
                fs, fill_s, lim_s = nextday_limit_fill(o1, h1, l1, atr1, ev_i, best_p_adj.enter_abs, -1, best_p_adj.entry_discount)
                next_day_filled_buy = bool(fb)
                next_day_filled_sell = bool(fs)

                # Keep "best_*" always informative: projected limit if no fill, filled price if filled
                best_buy_value = float(fill_b) if fb else (float(lim_b) if np.isfinite(lim_b) else (float(projected_buy_limit) if np.isfinite(projected_buy_limit) else np.nan))
                best_sell_value = float(fill_s) if fs else (float(lim_s) if np.isfinite(lim_s) else (float(projected_sell_limit) if np.isfinite(projected_sell_limit) else np.nan))

                if final_signal == "buy":
                    next_day_filled = bool(fb)
                    limit_price = float(lim_b) if np.isfinite(lim_b) else np.nan
                    entry_ref_price = float(fill_b) if fb else (float(lim_b) if np.isfinite(lim_b) else float(o1))
                elif final_signal == "sell":
                    next_day_filled = bool(fs)
                    limit_price = float(lim_s) if np.isfinite(lim_s) else np.nan
                    entry_ref_price = float(fill_s) if fs else (float(lim_s) if np.isfinite(lim_s) else float(o1))
                else:
                    # For hold rows, still expose a meaningful reference for risk columns
                    entry_ref_price = float(o1)

            # compute levels from reference price and ATR of next day (if available)
            atr_for_levels = float(atr[i+1]) if (i + 1) < len(g) and np.isfinite(atr[i+1]) else atr_i
            levels = compute_levels_from_atr(entry_ref_price, atr_for_levels, best_p_adj)

            results_apply.append({
                "Date": date_i,
                "ticker": tkr,
                "close": close_i,
                "atr": atr_i,

                "signal": final_signal,

                "score_0_100": float(score100),
                "score_ev": ev_i,

                "wf_auc_mean": wf_m["wf_auc_mean"],
                "wf_auc_std": wf_m["wf_auc_std"],
                "wf_acc_mean": wf_m["wf_acc_mean"],
                "wf_ap_mean": wf_m["wf_ap_mean"],
                "wf_logloss": wf_m["wf_logloss"],
                "wf_brier": wf_m["wf_brier"],
            "wf_fold_ranges": wf_m["wf_fold_ranges"],

                "ga_enter_abs": float(best_p_adj.enter_abs),
                "ga_exit_abs": float(best_p_adj.exit_abs),
                "ga_atr_mult": float(best_p_adj.atr_mult),
                "ga_rr_mult": float(best_p_adj.rr_mult),
                "ga_entry_discount": float(best_p_adj.entry_discount),
                "wf_quality": float(wf_quality),

                "ga_return_1y": float(stats_ga_train["total_return"]),
                "ga_mdd_1y": float(stats_ga_train["mdd"]),
                "ga_sharpe_1y": float(stats_ga_train["sharpe"]),
                "ga_trades_1y": float(stats_ga_train["n_trades"]),
                "ga_exposure_1y": float(stats_ga_train["exposure"]),
                "buyhold_return_1y": float(bh_ga),

                "test_return": float(te_stats.get("total_return", np.nan)),
                "test_mdd": float(te_stats.get("mdd", np.nan)),
                "test_sharpe": float(te_stats.get("sharpe", np.nan)),
                "test_n_trades": float(te_stats.get("n_trades", np.nan)),

                "next_day_filled": bool(next_day_filled),
                "next_day_filled_buy": bool(next_day_filled_buy),
                "next_day_filled_sell": bool(next_day_filled_sell),
                "projected_buy_limit": float(projected_buy_limit) if np.isfinite(projected_buy_limit) else np.nan,
                "projected_sell_limit": float(projected_sell_limit) if np.isfinite(projected_sell_limit) else np.nan,
                "limit_price_next_day": float(limit_price) if np.isfinite(limit_price) else np.nan,
                "best_buy_value": float(best_buy_value) if np.isfinite(best_buy_value) else np.nan,
                "best_sell_value": float(best_sell_value) if np.isfinite(best_sell_value) else np.nan,
                "entry_ref_price": float(entry_ref_price) if np.isfinite(entry_ref_price) else np.nan,

                "stop_abs": levels[0], "take_abs": levels[1], "stop_pct": levels[2], "take_pct": levels[3],
                "buy_entry": levels[4], "buy_stop": levels[5], "buy_take": levels[6],
                "sell_entry": levels[7], "sell_stop": levels[8], "sell_take": levels[9],

                "train_start": train_rng[0], "train_end": train_rng[1],
                "test_start": test_rng[0], "test_end": test_rng[1],
            })

        # summary latest (at last available close)
        last_i = len(g) - 1
        latest_date  = pd.to_datetime(dates[last_i])
        latest_close = float(c[last_i])
        latest_atr   = float(atr[last_i]) if np.isfinite(atr[last_i]) else np.nan
        latest_z     = float(score_ev[last_i]) if np.isfinite(score_ev[last_i]) else np.nan
        latest_ma    = float(ma[last_i]) if np.isfinite(ma[last_i]) else np.nan
        latest_sl    = float(sl[last_i]) if np.isfinite(sl[last_i]) else np.nan

        signal_eod_latest = make_signal_eod(latest_z, best_p_adj.enter_abs, latest_close, latest_ma, latest_sl)
        final_signal_latest = signal_eod_latest
        latest_quality = 0.60 * wf_quality + 0.40 * compute_quality_factor(float(te_stats.get("sharpe", np.nan)), float(te_stats.get("total_return", np.nan)), float(stats_ga_train.get("n_trades", np.nan)))
        latest_recent_scores = score_ev[max(0, last_i-503): last_i+1]
        latest_score100 = score_0_100_from_ev(latest_z, latest_recent_scores, latest_quality)

        results_summary.append({
            "ticker": tkr,
            "feat_count_used": int(len(feat_cols)),
            "wf_auc_mean": wf_m["wf_auc_mean"],
            "wf_auc_std": wf_m["wf_auc_std"],
            "wf_acc_mean": wf_m["wf_acc_mean"],
            "wf_ap_mean": wf_m["wf_ap_mean"],
            "wf_logloss": wf_m["wf_logloss"],
            "wf_brier": wf_m["wf_brier"],
            "wf_fold_ranges": wf_m["wf_fold_ranges"],

            "ga_enter_abs": float(best_p_adj.enter_abs),
            "ga_exit_abs": float(best_p_adj.exit_abs),
            "ga_atr_mult": float(best_p_adj.atr_mult),
            "ga_rr_mult": float(best_p_adj.rr_mult),
            "ga_entry_discount": float(best_p_adj.entry_discount),
            "wf_quality": float(wf_quality),
            "ga_return_1y": float(stats_ga_train["total_return"]),
            "ga_mdd_1y": float(stats_ga_train["mdd"]),
            "ga_sharpe_1y": float(stats_ga_train["sharpe"]),
            "ga_trades_1y": float(stats_ga_train["n_trades"]),
            "ga_exposure_1y": float(stats_ga_train["exposure"]),
            "ga_fitness_1y": float(fit_ga),
            "buyhold_return_1y": float(bh_ga),

            "test_return": float(te_stats.get("total_return", np.nan)),
            "test_mdd": float(te_stats.get("mdd", np.nan)),
            "test_sharpe": float(te_stats.get("sharpe", np.nan)),
            "test_n_trades": float(te_stats.get("n_trades", np.nan)),

            "latest_date": latest_date,
            "latest_close": latest_close,
            "latest_atr": latest_atr,
            "latest_score_ev": latest_z,
            "signal": final_signal_latest,
            "score_0_100": float(latest_score100),

            "train_start": train_rng[0], "train_end": train_rng[1],
            "test_start": test_rng[0], "test_end": test_rng[1],
        })

    if not results_summary:
        print("\nNenhum ticker gerou resultado.")
        if reasons:
            print("\n[DIAGNÓSTICO - motivos de descarte]")
            for k, v in sorted(reasons.items(), key=lambda x: -x[1]):
                print(f"  {k}: {v}")
        return

    summary_df = pd.DataFrame(results_summary).copy()
    summary_df["latest_date"] = pd.to_datetime(summary_df["latest_date"], errors="coerce")
    summary_df = summary_df.sort_values(["score_0_100", "ga_fitness_1y"], ascending=[False, False])

    apply_df = pd.DataFrame(results_apply).copy()
    apply_df["Date"] = pd.to_datetime(apply_df["Date"], errors="coerce")
    apply_df = apply_df.sort_values(["ticker", "Date"], ascending=[True, False])

    signals_cols = ["Date", "ticker", "close", "signal", "score_0_100", "best_buy_value", "best_sell_value", "buy_stop", "buy_take", "sell_stop", "sell_take", "stop_pct", "take_pct", "atr"]
    debug_cols = ["Date", "ticker", "close", "signal", "score_0_100", "score_ev", "wf_auc_mean", "wf_auc_std", "wf_acc_mean", "wf_ap_mean", "wf_logloss", "wf_brier", "wf_fold_ranges", "wf_quality", "ga_enter_abs", "ga_exit_abs", "ga_atr_mult", "ga_rr_mult", "ga_entry_discount", "ga_return_1y", "ga_mdd_1y", "ga_sharpe_1y", "ga_trades_1y", "ga_exposure_1y", "buyhold_return_1y", "test_return", "test_mdd", "test_sharpe", "test_n_trades", "best_buy_value", "best_sell_value", "projected_buy_limit", "projected_sell_limit", "next_day_filled", "next_day_filled_buy", "next_day_filled_sell", "limit_price_next_day", "entry_ref_price", "stop_abs", "take_abs", "train_start", "train_end", "test_start", "test_end"]
    signals_df = apply_df.reindex(columns=signals_cols)
    debug_df = apply_df.reindex(columns=debug_cols)

    out_xlsx = f"{OUTPUT_DIR}apply_PER_TICKER_WFGA_intraday__H{FWD_H}__APPLY{APPLY_DAYS}D__v2.xlsx"
    out_apply_csv = f"{OUTPUT_DIR}apply_last_{APPLY_DAYS}d__H{FWD_H}__v2.csv"

    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        signals_df.to_excel(writer, sheet_name="signals", index=False)
        debug_df.to_excel(writer, sheet_name="debug_metrics", index=False)

    apply_df.to_csv(out_apply_csv, index=False, encoding="utf-8")

    print(f"\n[OK] Saved: {out_xlsx}")
    print(f"[OK] Saved CSV apply: {out_apply_csv}")

    print("\n[SINAIS - latest (final)]")
    print(summary_df["signal"].value_counts(dropna=False))


    if reasons:
        print("\n[DIAGNÓSTICO - motivos de descarte]")
        for k, v in sorted(reasons.items(), key=lambda x: -x[1]):
            print(f"  {k}: {v}")


if __name__ == "__main__":
    run()


[FEATS] numeric candidates: 28 (using up to 60)
[TICKERS] total: 81

PROG | TICKER     |    AUC  AUCsd   ACC    AP |   GAret   GAmdd   GAsh   TR   EX      BH ||   TEret   TEmdd   TEsh TEtr | TRAIN                 TEST                 
---------------------------------------------------------------------------------------------------------------------------------------------------------------------
   1 | ABCB4.SA   |  0.609  0.056 0.527 0.597 |    16.5%    -7.7%   1.49    8  12%    23.7% ||     0.9%    -6.9%   0.06    5 | 2011-02-24→2023-01-23 2023-01-24→2026-01-12
   2 | ABEV3.SA   |  0.692  0.076 0.640 0.715 |    -1.4%    -2.2%  -0.91    1   1%    -2.0% ||     7.9%    -2.4%   0.55    9 | 2008-12-29→2022-08-18 2022-08-19→2026-01-12
   3 | AGRO3.SA   |  0.523  0.096 0.520 0.571 |    -8.7%    -8.7%  -1.77    5   1%    -6.4% ||    -8.7%   -10.0%  -0.47   16 | 2010-02-19→2022-11-10 2022-11-11→2026-01-12
   4 | ALUP11.SA  |  0.635  0.123 0.557 0.694 |     3.7%    -1.5%   0.91    2   3%    

NameError: name 'apply_df' is not defined